# 01 - Mayo Data Preparation and Sparse-View CT Degradation

This notebook builds the fixed dataset used throughout the sparse-view CT reconstruction pipeline. It loads the Mayo slices, resizes and normalizes them, applies the parallel-beam CT forward operator, adds controlled Gaussian measurement noise, and saves the resulting clean images and degraded sinograms. No inverse or reconstruction method is executed in this stage.


## Environment, Paths, and Fixed Parameters

The notebook is configured for execution in Google Colab with data stored on Google Drive. The project root defines the location of the Mayo dataset, the IPPy package, and the processed output directory. The acquisition parameters are fixed here so that all downstream reconstruction notebooks use the same image size, angle sets, detector size, geometry, and noise level.


In [ ]:
!pip install astra-toolbox

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import json
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/LM_INFORMATICA/COMPUTATIONAL_IMAGING")
MAYO_DIR = PROJECT_ROOT / "Mayo"
TRAIN_DIR = MAYO_DIR / "train"
TEST_DIR = MAYO_DIR / "test"
IPPY_DIR = PROJECT_ROOT / "IPPy"
OUTPUT_DIR = PROJECT_ROOT / "processed"

sys.path.append(str(PROJECT_ROOT))

from IPPy import operators, utilities
from IPPy.utilities import load_image, normalize

SEED = 42
IMAGE_SIZE = 256
ANGLE_COUNTS = (180, 90, 60, 45)
DETECTOR_SIZE = 256
GEOMETRY = "parallel"
NOISE_LEVEL = 0.005

torch.manual_seed(SEED)
np.random.seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Mayo directory:", MAYO_DIR)
print("IPPy directory:", IPPY_DIR)
print("Output directory:", OUTPUT_DIR)
print("Angle counts:", ANGLE_COUNTS)
print("Noise level:", NOISE_LEVEL)


## Dataset Loading and Preprocessing

Each Mayo slice is loaded as a tensor and converted to the `[batch, channels, height, width]` layout required by PyTorch interpolation. The images are resized to `256 x 256` and normalized to a stable intensity range before projection. This produces clean ground-truth tensors with shape `[N, 1, 256, 256]` for every patient file.


In [ ]:
def collect_png_paths(split_dir: Path) -> list[Path]:
    return sorted(split_dir.glob("*/*.png"))


def ensure_4d_grayscale(image: torch.Tensor) -> torch.Tensor:
    image = torch.as_tensor(image).float()

    if image.ndim == 2:
        image = image.unsqueeze(0).unsqueeze(0)
    elif image.ndim == 3:
        image = image.unsqueeze(0) if image.shape[0] == 1 else image.permute(2, 0, 1).unsqueeze(0)

    return image



def load_and_preprocess_image(image_path: Path) -> torch.Tensor:
    image = load_image(str(image_path))
    image = ensure_4d_grayscale(image)
    image = F.interpolate(
        image,
        size=(IMAGE_SIZE, IMAGE_SIZE),
        mode="bilinear",
        align_corners=False,
    )
    return normalize(image).clamp(0.0, 1.0).to(torch.float32)


def load_clean_batch(paths: list[Path]) -> torch.Tensor:
    return torch.cat([load_and_preprocess_image(path) for path in paths], dim=0)


split_paths = {
    "train": collect_png_paths(TRAIN_DIR),
    "test": collect_png_paths(TEST_DIR),
}

for split_name, paths in split_paths.items():
    print(f"{split_name}: {len(paths)} images")

sample_clean = load_and_preprocess_image(split_paths["train"][0])
print("Sample clean shape:", tuple(sample_clean.shape))
print("Sample clean range:", float(sample_clean.min()), float(sample_clean.max()))

## Forward CT Model and Patient-Level Export Functions

The degradation model simulates parallel-beam sparse-view CT. For each angle configuration, the CT projector maps every clean slice to a sinogram, and additive Gaussian noise with fixed relative level `0.005` is applied in the measurement domain. The processed data are saved as one `.pt` file per Mayo patient, preserving a direct correspondence between each output file and its original `Cxxx` patient identifier.


In [ ]:
projectors = {
    str(n_views): operators.CTProjector(
        img_shape=(IMAGE_SIZE, IMAGE_SIZE),
        angles=np.linspace(0.0, np.pi, n_views),
        det_size=DETECTOR_SIZE,
        geometry=GEOMETRY,
    )
    for n_views in ANGLE_COUNTS
}


@torch.no_grad()
def make_noisy_sinograms(clean_batch: torch.Tensor) -> tuple[dict[str, torch.Tensor], dict[str, float]]:
    sinograms = {}
    relative_noise = {}

    for key, projector in projectors.items():
        noisy_measurements = []
        sample_noise_levels = []

        for image in clean_batch:
            x = image.unsqueeze(0)
            y = projector(x)
            noise = utilities.gaussian_noise(y, noise_level=NOISE_LEVEL)

            noisy_measurements.append((y + noise).detach().cpu().to(torch.float32))
            sample_noise_levels.append(float(torch.norm(noise.detach().cpu()) / torch.norm(y.detach().cpu())))

        sinograms[key] = torch.cat(noisy_measurements, dim=0)
        relative_noise[key] = float(np.mean(sample_noise_levels))

    return sinograms, relative_noise


def group_paths_by_patient(paths: list[Path]) -> dict[str, list[Path]]:
    patient_paths = {}
    for path in paths:
        patient_id = path.parent.name
        patient_paths.setdefault(patient_id, []).append(path)
    return patient_paths


def save_patient_file(split_name: str, patient_id: str, patient_image_paths: list[Path]) -> dict:
    clean = load_clean_batch(patient_image_paths)
    sinograms, relative_noise = make_noisy_sinograms(clean)

    patient_path = OUTPUT_DIR / split_name / f"{patient_id}.pt"
    patient_path.parent.mkdir(parents=True, exist_ok=True)
    source_paths = [str(path) for path in patient_image_paths]

    payload = {
        "clean": clean.detach().cpu().to(torch.float32),
        "sinograms": sinograms,
        "source_paths": source_paths,
        "metadata": {
            "split": split_name,
            "patient_id": patient_id,
            "num_samples": len(patient_image_paths),
            "image_size": IMAGE_SIZE,
            "angle_counts": list(ANGLE_COUNTS),
            "detector_size": DETECTOR_SIZE,
            "geometry": GEOMETRY,
            "noise_level": NOISE_LEVEL,
            "relative_noise": relative_noise,
        },
    }
    torch.save(payload, patient_path)

    return {
        "path": str(patient_path.relative_to(OUTPUT_DIR)),
        "patient_id": patient_id,
        "num_samples": len(patient_image_paths),
        "source_paths": source_paths,
        "relative_noise": relative_noise,
    }


def save_split_patients(split_name: str, paths: list[Path]) -> list[dict]:
    patient_paths = group_paths_by_patient(paths)
    patient_records = []

    for patient_id, patient_image_paths in tqdm(
        sorted(patient_paths.items()),
        total=len(patient_paths),
        desc=f"{split_name} patients",
    ):
        patient_records.append(save_patient_file(split_name, patient_id, patient_image_paths))

    return patient_records


sample_sinograms, sample_noise = make_noisy_sinograms(sample_clean)
for key, value in sample_sinograms.items():
    print(f"{key} views: {tuple(value.shape)} | relative noise {sample_noise[key]:.6f}")


## Generate the Processed Dataset

This cell generates the full processed dataset for the train and test splits. Existing processed files for those splits are removed before export so that outputs from older formats cannot be mixed with the current patient-level representation. The manifest is written after all patient files have been saved and checked against the expected image counts.


In [ ]:
for split_name in split_paths:
    split_output_dir = OUTPUT_DIR / split_name
    if split_output_dir.exists():
        shutil.rmtree(split_output_dir)

manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.unlink(missing_ok=True)

manifest = {
    "project_root": str(PROJECT_ROOT),
    "mayo_dir": str(MAYO_DIR),
    "output_dir": str(OUTPUT_DIR),
    "image_size": IMAGE_SIZE,
    "angle_counts": list(ANGLE_COUNTS),
    "detector_size": DETECTOR_SIZE,
    "geometry": GEOMETRY,
    "noise_level": NOISE_LEVEL,
    "normalization": "IPPy.utilities.normalize after IPPy.utilities.load_image and resize",
    "storage": "one PyTorch .pt file per Mayo patient",
    "splits": {},
}

for split_name, paths in split_paths.items():
    patient_records = save_split_patients(split_name, paths)
    manifest["splits"][split_name] = {
        "num_images": len(paths),
        "num_patients": len(patient_records),
        "patients": patient_records,
    }

for split_name, split_info in manifest["splits"].items():
    patient_records = split_info["patients"]
    expected_images = split_info["num_images"]
    expected_patients = split_info["num_patients"]
    saved_files = sorted((OUTPUT_DIR / split_name).glob("*.pt"))
    listed_images = sum(record["num_samples"] for record in patient_records)
    missing_files = [record["path"] for record in patient_records if not (OUTPUT_DIR / record["path"]).exists()]

    if len(saved_files) != expected_patients:
        raise RuntimeError(f"Saved patient-file count mismatch for {split_name}: {len(saved_files)} != {expected_patients}")
    if listed_images != expected_images:
        raise RuntimeError(f"Image count mismatch for {split_name}: {listed_images} != {expected_images}")
    if missing_files:
        raise RuntimeError(f"Missing patient files for {split_name}: {missing_files[:5]}")

manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Saved manifest:", manifest_path)


## Visual Sanity Check and Output Summary

A saved patient file is reloaded and one clean slice is displayed together with its four noisy sparse-view sinograms. The purpose of this cell is to verify the saved measurement data and summarize the processed dataset structure.


In [ ]:
preview_split = "train"
preview_patient_path = OUTPUT_DIR / manifest["splits"][preview_split]["patients"][0]["path"]
preview_payload = torch.load(preview_patient_path, map_location="cpu")

clean_preview = preview_payload["clean"][0, 0]
sinogram_previews = preview_payload["sinograms"]

fig, axes = plt.subplots(1, 1 + len(ANGLE_COUNTS), figsize=(4 * (1 + len(ANGLE_COUNTS)), 4), constrained_layout=True)
axes[0].imshow(clean_preview, cmap="gray", vmin=0.0, vmax=1.0)
axes[0].set_title("Clean 256x256")
axes[0].axis("off")

for ax, n_views in zip(axes[1:], ANGLE_COUNTS):
    key = str(n_views)
    ax.imshow(sinogram_previews[key][0, 0], cmap="gray", aspect="auto")
    ax.set_title(f"{key} views")
    ax.set_xlabel("Detector")
    ax.set_ylabel("Angle")

plt.show()

print("Processed dataset root:", OUTPUT_DIR)
print("Manifest:", OUTPUT_DIR / "manifest.json")
for split_name, split_info in manifest["splits"].items():
    print(f"{split_name}: {split_info['num_images']} images, {split_info['num_patients']} patient files")


## Loading Contract for Downstream Notebooks

Each processed patient file is a PyTorch dictionary with clean ground-truth images, one noisy sinogram tensor for each sparse-view setting, source file paths, and metadata. Downstream notebooks use this contract directly so that every reconstruction method is evaluated on the same fixed degraded measurements.


In [ ]:
contract_split = "train"
contract_patient_path = OUTPUT_DIR / manifest["splits"][contract_split]["patients"][0]["path"]
contract_payload = torch.load(contract_patient_path, map_location="cpu")

clean = contract_payload["clean"]
sinogram_180 = contract_payload["sinograms"]["180"]
sinogram_90 = contract_payload["sinograms"]["90"]
sinogram_60 = contract_payload["sinograms"]["60"]
sinogram_45 = contract_payload["sinograms"]["45"]
source_paths = contract_payload["source_paths"]
metadata = contract_payload["metadata"]

print("Patient file:", contract_patient_path)
print("clean:", tuple(clean.shape))
print("sinogram_180:", tuple(sinogram_180.shape))
print("sinogram_90:", tuple(sinogram_90.shape))
print("sinogram_60:", tuple(sinogram_60.shape))
print("sinogram_45:", tuple(sinogram_45.shape))
print("patient_id:", metadata["patient_id"])
print("first source path:", source_paths[0])
